# Evaluación inicial de síntesis de voz para Memi

Notebook de planificación técnica para explorar síntesis de voz personalizada como herramienta de apoyo para personas laringectomizadas. No es asesoramiento clínico.

## Alcance y seguridad

Memi es un proyecto de apoyo, no un dispositivo médico y no reemplaza el trabajo con el/la fonoaudiólogo/a. Esta notebook no recolecta, carga, reproduce ni genera audios por sí sola. Las celdas que instalan paquetes, descargan modelos y sintetizan audio están desactivadas por defecto.

La voz es un dato personal. No uses grabaciones propias o de otra persona sin consentimiento informado, explícito y revocable. Antes de trabajar con datos reales, definir quién tendrá acceso, dónde se procesarán, cómo se borrarán y cómo se evitará su sincronización o copia de seguridad no autorizada. Mantén audios en `audios/` y modelos en `modelos/`; ambas rutas están excluidas en `.gitignore`. `.gitignore` reduce el riesgo de un `git add` accidental, pero no protege datos compartidos por otros medios ni elimina archivos que ya estén versionados.

## Licencias

XTTS-v2 se publica bajo la Coqui Public Model License (CPML), una licencia propia que no debe asumirse equivalente a una licencia open source estándar. Revisa la licencia y sus condiciones para el uso previsto antes de descargar o usar el modelo. El código de Memi, los pesos del modelo, las dependencias y las grabaciones tienen licencias y derechos distintos.

Model card: https://huggingface.co/coqui/XTTS-v2
Licencia CPML: https://coqui.ai/cpml

## 1. Protección de archivos sensibles con `.gitignore`

El archivo `.gitignore` del repositorio excluye explícitamente `/audios/` y `/modelos/`, además de extensiones habituales de audio y archivos de pesos. Esta comprobación local solo inspecciona el texto del archivo; no lee ningún audio ni modelo.

In [ ]:
from pathlib import Path

ignore_text = Path(".gitignore").read_text(encoding="utf-8")
assert "/audios/" in ignore_text
assert "/modelos/" in ignore_text
print("OK: las carpetas audios/ y modelos/ figuran excluidas en .gitignore")

## 2. Alcance del proyecto en el README

El README debe comunicar el alcance de apoyo, no médico, y que el proyecto no reemplaza el trabajo con el/la fonoaudiólogo/a. La formulación acordada se verifica aquí para evitar que quede fuera de la documentación principal.

In [ ]:
readme_text = Path("README.md").read_text(encoding="utf-8")
assert "no es un dispositivo médico" in readme_text
assert "no reemplaza el trabajo con su fonoaudiólogo" in readme_text
print("OK: alcance no médico documentado")

## 3. Dependencias (opt-in)

XTTS y PyTorch pueden requerir versiones específicas y una instalación considerable. Primero revisa los requisitos vigentes del proyecto Coqui TTS y el entorno de ejecución. La siguiente celda instala dependencias únicamente si cambias la bandera a `True`; no la ejecutes antes de revisar licencia, compatibilidad del entorno y necesidad real.

In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch",
        "transformers",
        "soundfile",
        "TTS",
        "huggingface_hub",
    ])
else:
    print("Instalación omitida. Revisa primero licencias y requisitos del entorno.")

## 4. Descargar XTTS-v2 en la carpeta ignorada (opt-in)

XTTS-v2 es un candidato, no una elección definitiva. Confirma la licencia CPML para el uso previsto y sus condiciones antes de continuar. Esta celda descarga archivos del modelo desde Hugging Face a `modelos/XTTS-v2`; no descarga ni procesa grabaciones personales. La ejecución permanece desactivada hasta que cambies ambas confirmaciones.

In [ ]:
MODEL_LICENSE_REVIEWED = False
MODEL_TERMS_ACCEPTED = False
DOWNLOAD_MODEL = False

model_dir = Path("modelos/XTTS-v2")

if MODEL_LICENSE_REVIEWED and MODEL_TERMS_ACCEPTED and DOWNLOAD_MODEL:
    from huggingface_hub import snapshot_download

    model_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="coqui/XTTS-v2",
        local_dir=str(model_dir),
    )
else:
    print("Descarga omitida. Revisa y acepta expresamente la licencia antes de habilitarla.")

## 5. Ejemplo de síntesis de voz (opt-in)

Para una prueba futura, coloca la referencia de voz solo en `audios/referencia.wav` después de contar con consentimiento informado y revocable de su titular. No subas el archivo ni lo adjuntes a incidencias o servicios externos. La muestra de texto siguiente es genérica. La bandera de consentimiento y la de generación están desactivadas; la salida es sintética y debe identificarse como tal al compartirla.

In [ ]:
SPEAKER_CONSENT_CONFIRMED = False
SYNTHESIS_APPROVED = False
speaker_wav_path = Path("audios/referencia.wav")
output_wav_path = Path("audios/prueba_generada.wav")

if (
    MODEL_LICENSE_REVIEWED
    and MODEL_TERMS_ACCEPTED
    and SPEAKER_CONSENT_CONFIRMED
    and SYNTHESIS_APPROVED
    and model_dir.is_dir()
    and speaker_wav_path.is_file()
):
    from TTS.api import TTS

    output_wav_path.parent.mkdir(parents=True, exist_ok=True)
    tts = TTS(
        model_path=str(model_dir),
        config_path=str(model_dir / "config.json"),
        gpu=False,
    )
    tts.tts_to_file(
        text="Hola, este es un mensaje de prueba generado con una voz sintética.",
        speaker_wav=str(speaker_wav_path),
        language="es",
        file_path=str(output_wav_path),
    )
    print(f"Audio generado localmente: {output_wav_path}")
else:
    print("Síntesis omitida. Requiere revisión de licencia, consentimiento y aprobación explícita.")

## 6. Guardar y reproducir el resultado

Los audios de prueba se guardan bajo `audios/`, excluida por Git. Reproducir audio puede revelar información personal a quienes estén cerca; hazlo solo en un entorno privado y con autorización.

In [ ]:
if output_wav_path.is_file():
    from IPython.display import Audio, display

    display(Audio(filename=str(output_wav_path)))
else:
    print("No hay audio generado para reproducir.")

## Próximos pasos

- Definir una política de gobernanza, consentimiento revocable, acceso, almacenamiento y borrado antes de cualquier prueba con voces reales.
- Revisar con asesoramiento adecuado si CPML permite el uso previsto; comparar alternativas con licencias de código y modelos compatibles con el objetivo open source del proyecto.
- Definir un protocolo de evaluación con personas usuarias y su fonoaudiólogo/a. Considerar inteligibilidad, similitud con la voz elegida por la persona, naturalidad, desempeño en español, latencia, accesibilidad y privacidad; no presentar resultados como evaluación clínica.
- Confirmar requisitos técnicos y compatibilidad del modelo en el equipo de destino antes de instalar dependencias o descargar pesos.
- Mantener audios y modelos fuera de Git. Antes de cada commit, revisar `git status` y `git diff --cached --stat`; `.gitignore` no quita del historial archivos que ya hayan sido agregados o publicados.